# RAG Step 4: Prompt Augmentation
Combine the retrieved context and the question into one grounded prompt.

In [ ]:
!pip install -q sentence-transformers chromadb

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
col = chromadb.PersistentClient(path="./chroma_store").get_collection("rag_demo")

def retrieve(question, k=3):
    q = model.encode([question], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q, n_results=k)
    return list(zip(res["documents"][0], [m["source"] for m in res["metadatas"][0]]))

## 1. A basic template

In [ ]:
TEMPLATE = '''Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
Answer:'''

def build_prompt(question, k=3):
    ctx = "\n".join(f"[{i+1}] ({src}) {txt}" for i, (txt, src) in enumerate(retrieve(question, k)))
    return TEMPLATE.format(context=ctx, question=question)

print(build_prompt("What does a vector database do?"))

## 2. Citations
Ask the model to cite `[n]` markers so answers can be verified.

In [ ]:
CITED = TEMPLATE.replace("Answer:", "Cite the sources you used like [1], [2].\nAnswer:")
print(CITED.format(context="[1] (demo) RAG uses retrieval.", question="What does RAG use?"))

## 3. Context budget
Trim context to a rough token limit (about 4 characters per token).

In [ ]:
def trim_context(chunks, max_tokens=300):
    out, used = [], 0
    for txt, src in chunks:
        cost = len(txt) // 4
        if used + cost > max_tokens: break
        out.append((txt, src)); used += cost
    return out

print(len(trim_context(retrieve("What is RAG?", 5), max_tokens=60)), "chunks kept")

## Takeaways
A clear instruction, labelled context and a fallback answer ("I don't know") are the key ingredients.